# FlyWise saved-model evaluation
Run each cell in order. No model retraining. Requires full flight_data_final.csv and the trusted team model files. Outputs are actual results only after a successful run. Cleaning and feature logic were checked against FlyWise(short-for_extracting_model).ipynb; dataset/model consistency still needs execution.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
%pip install -q pandas numpy scikit-learn==1.6.1 xgboost joblib

In [ ]:
from pathlib import Path
import subprocess, sys, json
project = Path("/content/drive/MyDrive/flywise")
output = project / "analytics/outputs/model_evaluation"
output.mkdir(parents=True, exist_ok=True)
command = [sys.executable, str(project/"analytics/evaluate_model.py"), "--data", str(project/"flight_data_final.csv"), "--artifacts", str(project/"flywise final model"), "--feature-spec", str(project/"analytics/evaluation_config.json"), "--out", str(output)]
with open(output/"execution.log", "w") as log:
    result = subprocess.run(command, stdout=log, stderr=subprocess.STDOUT)
print((output/"execution.log").read_text())
if result.returncode:
    raise RuntimeError("Evaluation failed: inspect execution.log")

In [ ]:
for name in ["metrics.json", "baseline_metrics.json", "run_summary.json"]:
    print(name)
    print((output/name).read_text())
import pandas as pd
display(pd.read_csv(output/"confusion_matrix.csv", index_col=0))
original = json.loads((project/"flywise final model/xgb_metrics.json").read_text())
actual = json.loads((output/"metrics.json").read_text())
for old,new in {"Accuracy":"accuracy","Precision":"precision","Recall":"recall","F1":"f1","ROC-AUC":"roc_auc"}.items():
    print(new, "original:", original[old], "rerun:", actual[new], "difference:", actual[new]-original[old])